# Candidate Clustering, Multi-DM Sifting, Harmonics & TEMPO2 Predictor Export

Demonstrates `pyloki.io.cands`:
1. Single-DM constant-phase-invariant transitive clustering & self-harmonic family flagging.
2. Multi-DM `SurveyCandidateGroups` with `DM=0` RFI filtering & tiered rational `HarmonicMatcher`.
3. Segment ephemeris export (JSON/CSV) and adaptive TEMPO2 `t2pred.dat` Chebyshev predictor export for arbitrary-order trajectories (including snap & crackle).

In [1]:
import tempfile
from pathlib import Path

import numpy as np

from pyloki.io.cands import (
    SECONDS_PER_DAY,
    CandidateClusterOptions,
    CandidateCollection,
    HarmonicMatcher,
    SurveyCandidateGroups,
    build_phase_model,
    cluster_candidates_single_dm,
    evaluate_tempo2_predictor,
    export_segment_ephemeris,
    export_tempo2_predictor,
)
from pyloki.utils.misc import C_VAL


def make_collection(
    dm: float,
    freqs: list[float],
    accels: list[float],
    snrs: list[float],
) -> CandidateCollection:
    f0 = np.asarray(freqs, dtype=np.float64)
    acc = np.asarray(accels, dtype=np.float64)
    n = f0.size
    return CandidateCollection(
        ids=np.arange(n, dtype=np.uint64),
        param_sets=np.column_stack([acc, f0]),
        param_names=("accel", "freq"),
        snr=np.asarray(snrs, dtype=np.float64),
        width=np.full(n, 4, dtype=np.int64),
        nbins=np.full(n, 64, dtype=np.int64),
        f0=f0,
        f1=-f0 * acc / C_VAL,
        f2=np.zeros(n, dtype=np.float64),
        dm=np.full(n, dm, dtype=np.float64),
    )

In [2]:
# 1. Single-DM phase-track transitive clustering + self-harmonic flagging
single_dm = make_collection(
    dm=238.0,
    freqs=[100.0, 100.00005, 100.00010, 200.0],
    accels=[10.0, 10.0, 10.0, 10.0],
    snrs=[20.0, 14.0, 12.0, 15.0],
)
cl = cluster_candidates_single_dm(
    single_dm,
    tobs=1000.0,
    options=CandidateClusterOptions(
        grouping="transitive",
        flag_harmonics=True,
        remove_harmonics=True,
    ),
)
print(
    f"Clusters: {len(cl.representatives)}, counts={cl.member_counts.tolist()}, "
    f"selected={cl.selected.tolist()}"
)

Clusters: 2, counts=[3, 1], selected=[True, False]


In [3]:
# 2. Multi-DM sifting + DM=0 RFI rejection + known-pulsar harmonic matching
known_psr_p = 11.5632e-3
known_psr_f = 1.0 / known_psr_p
merged = CandidateCollection.merge(
    [
        make_collection(0.0, [50.0], [0.0], [25.0]),
        make_collection(
            238.0,
            [50.001, known_psr_f, 173.456],
            [0.0, 1.0, 2.0],
            [18.0, 12.0, 9.0],
        ),
        make_collection(
            242.0,
            [49.999, known_psr_f, 173.457],
            [0.0, 1.0, 2.0],
            [15.0, 22.0, 16.0],
        ),
    ]
)
groups = SurveyCandidateGroups.from_collection(merged, rel_freq_tol=1e-3)
groups.filter_dm0_and_span(rfi_ratio=0.5)
groups.tag_known_harmonics([("J1748-2446A", known_psr_p, "strong")])
for grp in groups.groups:
    print(
        f"Group {grp.group_id}: f0={grp.f0_peak:.3f} Hz, DM_peak={grp.dm_peak:.1f}, "
        f"S/N={grp.peak_snr:.1f}, disposition={grp.disposition}"
    )

matcher = HarmonicMatcher([("J1748-2446A", known_psr_p, "strong")])
hm = matcher.match_period(known_psr_p * (13.0 / 11.0))
print(f"Harmonic match: {hm.pulsar_name} at {hm.k}/{hm.m} (err={hm.rel_err:.2e})")

Group 0: f0=50.000 Hz, DM_peak=0.0, S/N=25.0, disposition=rfi
Group 1: f0=86.481 Hz, DM_peak=242.0, S/N=22.0, disposition=known_harmonic
Group 2: f0=173.457 Hz, DM_peak=242.0, S/N=16.0, disposition=new
Harmonic match: J1748-2446A at 13/11 (err=0.00e+00)


In [4]:
# 3. Arbitrary-order segment ephemeris & TEMPO2 t2pred.dat predictor export
ref_mjd = 60000.0
t_end = 1800.0
taylor_params = [1e-7, -2e-5, 3e-3, -15.0, 200.0]  # [crackle, snap, jerk, accel, freq]
with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    rows = export_segment_ephemeris(
        tmp / "ephem.json",
        model="taylor",
        params=taylor_params,
        ref_mjd=ref_mjd,
        t_start=0.0,
        t_end=t_end,
        n_segments=8,
        fmt="json",
    )
    pred_path = tmp / "t2pred.dat"
    info = export_tempo2_predictor(
        pred_path,
        model="taylor",
        params=taylor_params,
        ref_mjd=ref_mjd,
        mjd_start=ref_mjd,
        mjd_end=ref_mjd + t_end / SECONDS_PER_DAY,
    )
    t_test = np.linspace(0.0, t_end, 100)
    phi_true = build_phase_model("taylor", taylor_params)(t_test)
    phi_pred = evaluate_tempo2_predictor(pred_path, ref_mjd + t_test / SECONDS_PER_DAY)
    print(
        f"Exported {len(rows)} segments and {info['n_blocks']} TEMPO2 block(s); "
        f"max phase err = {float(np.max(np.abs(phi_pred - phi_true))):.2e} cycles"
    )

Exported 8 segments and 1 TEMPO2 block(s); max phase err = 6.27e-05 cycles
